# Semana 10 — Ruido, Decoherencia y Corrección de Errores
## Por qué los qubits reales no son perfectos

**Objetivo:** Entender los modelos de ruido cuántico (bit-flip, phase-flip, depolarizing), la decoherencia, y los fundamentos de la corrección de errores cuánticos con el código de repetición y el código de Steane.

**Hito verificable:** Simulas el código de corrección de errores de 3 qubits (bit-flip), demuestras que corrige errores con p<0.5, y graficas la tasa de error lógico vs. p físico.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
print('Librerías importadas correctamente')

## Ejercicio 10.1 — Modelos de error cuántico
Los errores cuánticos se modelan como canales: operadores que actúan sobre la matriz de densidad.

In [ ]:
def canal_bit_flip(rho, p):
    """Canal de bit-flip: aplica X con probabilidad p."""
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    I = np.eye(2, dtype=complex)
    return (1 - p) * rho + p * (X @ rho @ X.conj().T)

def canal_phase_flip(rho, p):
    """Canal de phase-flip: aplica Z con probabilidad p."""
    Z = np.array([[1, 0], [0, -1]], dtype=complex)
    return (1 - p) * rho + p * (Z @ rho @ Z.conj().T)

def canal_depolarizing(rho, p):
    """Canal depolarizante: mezcla con la identidad."""
    I = np.eye(2, dtype=complex)
    return (1 - p) * rho + p * I/2

def fidelidad(rho1, rho2):
    """Fidelidad entre dos matrices de densidad."""
    sqrt_rho1 = np.linalg.matrix_power(rho1, 1)  # simplificado para estados puros
    return np.real(np.trace(rho1 @ rho2))

# Estado |+⟩ = (|0⟩+|1⟩)/√2
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
rho_plus = np.outer(ket_plus, ket_plus.conj())

ps = [0.0, 0.1, 0.3, 0.5, 0.9, 1.0]
print(f'{"p":>6}  {"Bit-flip F":>12}  {"Phase-flip F":>14}  {"Depolar F":>12}')
print('-' * 50)
for p in ps:
    rho_bf = canal_bit_flip(rho_plus, p)
    rho_pf = canal_phase_flip(rho_plus, p)
    rho_dep = canal_depolarizing(rho_plus, p)
    f_bf  = np.real(np.trace(rho_plus @ rho_bf))
    f_pf  = np.real(np.trace(rho_plus @ rho_pf))
    f_dep = np.real(np.trace(rho_plus @ rho_dep))
    print(f'{p:>6.1f}  {f_bf:>12.4f}  {f_pf:>14.4f}  {f_dep:>12.4f}')

print()
print('F=1: sin error. F<1: fidelidad degradada por el canal de ruido.')

## Ejercicio 10.2 — Código de corrección de bit-flip de 3 qubits
El código más simple: codificar 1 qubit lógico en 3 qubits físicos.

In [ ]:
def simular_codigo_repeticion(p_error, n_runs=100000):
    """
    Simula el código de corrección de bit-flip de 3 qubits.
    Codifica |0⟩L → |000⟩ y |1⟩L → |111⟩
    Corrección por mayoría de votos.
    Devuelve tasa de error lógico.
    """
    # Generar bit lógico (0 o 1) aleatorio
    logico = np.random.randint(0, 2, size=n_runs)
    
    # Codificar: 0 → [0,0,0], 1 → [1,1,1]
    fisicos = np.stack([logico, logico, logico], axis=1)  # (n_runs, 3)
    
    # Aplicar errores de bit-flip independientes
    errores = np.random.random((n_runs, 3)) < p_error
    fisicos_ruidosos = (fisicos + errores) % 2
    
    # Corrección: mayoría de votos
    corregido = (fisicos_ruidosos.sum(axis=1) >= 2).astype(int)
    
    # Tasa de error lógico
    tasa_error = np.mean(corregido != logico)
    return tasa_error

ps = np.linspace(0, 0.5, 50)
tasas_sin_codigo = ps  # Sin código, el error lógico = error físico
tasas_con_codigo = [simular_codigo_repeticion(p) for p in ps]
tasas_teoricas = [3*p**2*(1-p) + p**3 for p in ps]  # P(2 o 3 errores)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ps, tasas_sin_codigo, 'r-', label='Sin código (1 qubit)', linewidth=2)
ax.plot(ps, tasas_con_codigo, 'b.', label='Con código (3 qubits, simulado)', alpha=0.5)
ax.plot(ps, tasas_teoricas, 'g--', label='Con código (teórico: 3p²-2p³)', linewidth=2)
ax.axvline(1/3, color='gray', linestyle=':', label='Umbral p=1/3')
ax.set_xlabel('Probabilidad de error físico p')
ax.set_ylabel('Tasa de error lógico')
ax.set_title('Código de corrección de bit-flip (3 qubits)')
ax.legend()
plt.tight_layout()
plt.savefig('codigo_correccion_errores.png', dpi=100, bbox_inches='tight')
plt.show()

print('Para p < 1/3: el código MEJORA la tasa de error lógico.')
print('Para p > 1/3: el código EMPEORA (zona donde la mayoría está equivocada).')

## Ejercicio 10.3 — Simulación con ruido real en Qiskit Aer

In [ ]:
# Circuito de Bell sin ruido vs. con ruido depolarizante
def crear_bell():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

qc_bell = crear_bell()
sim_ideal = AerSimulator()

# Simulador ideal
counts_ideal = sim_ideal.run(qc_bell, shots=10000).result().get_counts()
print('Sin ruido:')
print(f'  {counts_ideal}')

# Modelo de ruido: error depolarizante en cada puerta
for p_noise in [0.01, 0.05, 0.1]:
    noise_model = NoiseModel()
    error_1q = depolarizing_error(p_noise, 1)
    error_2q = depolarizing_error(p_noise * 2, 2)
    noise_model.add_all_qubit_quantum_error(error_1q, ['h', 'x', 'rz', 'sx'])
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])
    
    sim_ruidoso = AerSimulator(noise_model=noise_model)
    counts_ruidoso = sim_ruidoso.run(qc_bell, shots=10000).result().get_counts()
    
    # Los estados 01 y 10 son error (en Bell ideal no deben aparecer)
    errores = counts_ruidoso.get('01', 0) + counts_ruidoso.get('10', 0)
    print(f'\np_noise={p_noise}: {counts_ruidoso}')
    print(f'  Estados de error (01, 10): {errores} ({100*errores/10000:.1f}%)')

## Ejercicio 10.4 — T1 y T2: tiempos de decoherencia

In [ ]:
# T1 = tiempo de relajación de amplitud (|1⟩ → |0⟩)
# T2 = tiempo de decoherencia de fase (pérdida de coherencia en la superposición)

# Modelo simplificado
def fidelidad_t1t2(t, T1, T2, estado='superposicion'):
    """Fidelidad de un estado tras un tiempo t con decoherencia."""
    if estado == 'excitado':
        # |1⟩ decae a |0⟩ con tiempo T1
        return np.exp(-t / T1)  # P(todavía en |1⟩)
    else:
        # |+⟩ pierde coherencia con tiempo T2
        # Las componentes off-diagonal decaen como e^(-t/T2)
        coherencia = np.exp(-t / T2) * np.exp(-t / (2*T1))
        return 0.5 * (1 + coherencia)  # Fidelidad con |+⟩

# Parámetros típicos de IBM Quantum (μs)
T1_ibm = 100  # microsegundos
T2_ibm = 150  # microsegundos
t_puerta = 0.1  # tiempo típico de una puerta: 100 ns = 0.1 μs

tiempos = np.linspace(0, 200, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# T1: relajación del estado excitado
f_t1 = fidelidad_t1t2(tiempos, T1_ibm, T2_ibm, 'excitado')
axes[0].plot(tiempos, f_t1, 'r-', linewidth=2)
axes[0].axvline(T1_ibm, color='gray', linestyle='--', label=f'T1={T1_ibm}μs')
axes[0].axhline(1/np.e, color='green', linestyle=':', alpha=0.7, label='1/e ≈ 0.37')
axes[0].set_xlabel('Tiempo (μs)')
axes[0].set_ylabel('P(estado excitado)')
axes[0].set_title('Relajación T1: |1⟩ → |0⟩')
axes[0].legend()

# T2: decoherencia de fase
f_t2 = fidelidad_t1t2(tiempos, T1_ibm, T2_ibm, 'superposicion')
axes[1].plot(tiempos, f_t2, 'b-', linewidth=2)
axes[1].axvline(T2_ibm, color='gray', linestyle='--', label=f'T2={T2_ibm}μs')
axes[1].axhline(0.5 + (1/(2*np.e)), color='green', linestyle=':', alpha=0.7)
axes[1].set_xlabel('Tiempo (μs)')
axes[1].set_ylabel('Fidelidad con |+⟩')
axes[1].set_title('Decoherencia T2: pérdida de fase en |+⟩')
axes[1].legend()

plt.tight_layout()
plt.savefig('decoherencia_t1t2.png', dpi=100, bbox_inches='tight')
plt.show()

n_puertas_en_T2 = T2_ibm / t_puerta
print(f'En T2={T2_ibm}μs caben aprox. {n_puertas_en_T2:.0f} puertas (si cada puerta tarda {t_puerta}μs).')
print('Por eso los circuitos útiles deben ser cortos (NISQ era).')

## Mini reto — Completar antes de avanzar a la Semana 11

Implementa en Qiskit el código de corrección de bit-flip de 3 qubits completo:
1. Codificación: CNOT de qubit 0 a qubits 1 y 2
2. Canal de ruido: aplicar X con probabilidad p a un qubit aleatorio
3. Síndrome: CNOT ancilla para detectar qué qubit tiene error
4. Corrección: medir las ancilla y aplicar X al qubit erróneo
5. Grafica tasa de error lógico vs. p para p ∈ [0, 0.5]

In [ ]:
# TU CÓDIGO AQUÍ
def codigo_bitflip_qiskit(p_error: float) -> dict:
    """Circuito completo del código de corrección de bit-flip."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 10

Antes de avanzar, comprueba que puedes:

- [ ] Implementar los canales de error (bit-flip, phase-flip, depolarizing) como matrices
- [ ] Simular el código de corrección de 3 qubits y verificar el umbral p=1/3
- [ ] Añadir ruido a un simulador Qiskit con NoiseModel
- [ ] Explicar la diferencia entre T1 y T2
- [ ] Calcular cuántas puertas caben en un tiempo T2 típico

## Reflexión (escribe aquí tu respuesta)

**¿Por qué la corrección de errores cuánticos es fundamentalmente más difícil que la clásica?**

*(tu respuesta aquí)*

**¿Qué es el umbral de tolerancia a fallos (fault-tolerance threshold)?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 11 — IBM Quantum en Hardware Real*